# Data Source Comparison & Metrics Assignment

Analyze data coverage from all ingestion sources and assign primary sources for each metric type.

**Purpose:**
1. Compare what data each API source provides
2. Identify gaps in coverage
3. Set primary + backup sources for each metric category
4. Track data quality and reliability

**Metric Categories:**
- Basic stats (points, yards, TDs)
- Air yards & target depth
- Red zone usage
- Pace & game script
- Coverage matchups
- O-line & pressure data
- Player tracking / Next Gen Stats
- Betting lines
- Ownership & ADP
- Coaching tendencies
- Historical similarity

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

SEASON = 2024
WEEK = 18

print(f"Data Source Comparison Analysis")
print(f"Season {SEASON}, Week {WEEK}")
print("="*70)

In [0]:
%sql
-- Compare record counts from each data source
SELECT 
  source,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(*) as total_records,
  COUNT(DISTINCT position) as positions_covered,
  ROUND(AVG(fantasy_points), 2) as avg_fantasy_points,
  MAX(ingested_at) as last_ingested
FROM main.fantasai.silver_weekly_stats
WHERE season = 2024 AND week = 18
GROUP BY source
ORDER BY unique_players DESC

In [0]:
%sql
-- See which positions each source covers well
SELECT 
  source,
  position,
  COUNT(DISTINCT player_id) as players,
  ROUND(AVG(fantasy_points), 2) as avg_points,
  ROUND(MAX(fantasy_points), 2) as max_points
FROM main.fantasai.silver_weekly_stats
WHERE season = 2024 AND week = 18
GROUP BY source, position
ORDER BY source, position

In [0]:
# Analyze what fields each source provides
print("Analyzing stat fields from each source...\n")

sources = spark.sql("""
  SELECT DISTINCT source 
  FROM main.fantasai.silver_weekly_stats 
  WHERE season = 2024 AND week = 18
""").collect()

for source_row in sources:
    source = source_row['source']
    print(f"\n{'='*70}")
    print(f"Source: {source}")
    print(f"{'='*70}")
    
    # Get sample record
    sample = spark.sql(f"""
        SELECT stats 
        FROM main.fantasai.silver_weekly_stats
        WHERE source = '{source}' AND season = 2024 AND week = 18
        LIMIT 1
    """).collect()
    
    if sample:
        import json
        stats_json = json.loads(sample[0]['stats'])
        fields = list(stats_json.keys())
        
        print(f"Total fields: {len(fields)}")
        print(f"\nAvailable fields:")
        
        # Categorize fields
        passing_fields = [f for f in fields if 'pass' in f.lower()]
        rushing_fields = [f for f in fields if 'rush' in f.lower()]
        receiving_fields = [f for f in fields if 'rec' in f.lower() or 'target' in f.lower()]
        
        if passing_fields:
            print(f"\n  Passing ({len(passing_fields)}): {', '.join(passing_fields[:10])}")
        if rushing_fields:
            print(f"  Rushing ({len(rushing_fields)}): {', '.join(rushing_fields[:10])}")
        if receiving_fields:
            print(f"  Receiving ({len(receiving_fields)}): {', '.join(receiving_fields[:10])}")
        
        # Show all unique fields
        print(f"\n  All fields: {', '.join(sorted(fields)[:30])}...")

## Recommended Primary Sources by Metric Category

Based on data availability testing:

### Basic Fantasy Stats
| Metric | Primary Source | Backup Source | Notes |
|--------|---------------|---------------|-------|
| Fantasy Points | nflverse | fantasy_data_pros | Most reliable |
| Passing Stats | nflverse | espn | Comprehensive |
| Rushing Stats | nflverse | fantasy_data_pros | Complete data |
| Receiving Stats | nflverse | espn | Target data included |

### Advanced Metrics (To Be Added)

#### Air Yards & Target Depth
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Air Yards | **Need: NextGenStats** | fantasypros | Not ingested |
| Target Depth | **Need: NextGenStats** | nflverse | Not ingested |
| Unrealized Air Yards | **Need: NextGenStats** | - | Not ingested |
| Weighted Opportunity | **Need: NextGenStats** | - | Not ingested |

#### Red Zone Usage
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| RZ Carries | **Need: NextGenStats** | fantasypros | Not ingested |
| Inside 5 Touches | **Need: NextGenStats** | - | Not ingested |
| End Zone Targets | **Need: NextGenStats** | - | Not ingested |
| Goal Line Snaps | **Need: NextGenStats** | - | Not ingested |

#### Pace & Game Script
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Plays per Game | nflverse | ✓ Available | ✓ Ready |
| Seconds per Snap | nflverse | ✓ Available | ✓ Ready |
| No-Huddle % | nflverse | ✓ Available | ✓ Ready |
| Pass Rate Over Exp | nflverse | ✓ Available | ✓ Ready |

#### Coverage Matchups
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| CB vs WR | **Need: PFF** | fantasypoints | Not ingested |
| Man vs Zone | **Need: PFF** | - | Not ingested |
| Slot Coverage | **Need: PFF** | - | Not ingested |
| Shadow Coverage | **Need: PFF** | - | Not ingested |

#### Offensive Line & Pressure
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Pass Block Win Rate | **Need: ESPN** | fantasypros | Not ingested |
| Pressure % | **Need: NextGenStats** | espn | Not ingested |
| Sack Rate | nflverse | espn | ✓ Ready |
| Adj Line Yards | **Need: ESPN** | - | Not ingested |

#### Player Tracking
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Separation | **Need: NextGenStats** | - | Not ingested |
| Speed | **Need: NextGenStats** | - | Not ingested |
| Route Depth | **Need: NextGenStats** | - | Not ingested |
| Time to Throw | **Need: NextGenStats** | - | Not ingested |

#### Betting & Market Data
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Point Spreads | **Need: The Odds API** | - | Not ingested |
| Over/Under | **Need: The Odds API** | - | Not ingested |
| Line Movement | **Need: The Odds API** | - | Not ingested |

#### Ownership & ADP
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Ownership % | **Need: Sleeper API** | - | Not ingested |
| Waiver Adds | **Need: Sleeper API** | - | Not ingested |
| Dynasty ADP | **Need: KeepTradeCut** | - | Not ingested |
| Best Ball ADP | **Need: Sleeper API** | - | Not ingested |

#### Coaching & Tendencies
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Run/Pass Splits | nflverse | ✓ Available | ✓ Ready |
| Red Zone Behavior | nflverse | ✓ Available | ✓ Ready |
| 4th Down Aggression | nflverse | ✓ Available | ✓ Ready |

#### Historical Similarity
| Metric | Primary Source | Backup Source | Status |
|--------|---------------|---------------|--------|
| Rookie Comparisons | nflverse historical | ✓ Available | ✓ Ready |
| Aging Curves | nflverse historical | ✓ Available | ✓ Ready |
| Injury Recovery | **Need: ESPN** | - | Not ingested |

## Missing Data Sources - Ingestion Notebooks To Create

### High Priority (Core Fantasy Metrics)

1. **❗ NextGenStats NFL.com Ingestion**
   - Air yards, target depth, separation, speed
   - Red zone touches, goal line snaps
   - Player tracking data
   - Endpoint: nextgenstats.nfl.com API
   - **Most valuable missing source**

2. **❗ The Odds API Ingestion**
   - Betting lines (spreads, over/under)
   - Line movement tracking
   - Implied game scripts
   - Free tier: 500 requests/month
   - Endpoint: https://the-odds-api.com

3. **❗ Sleeper API Ingestion**
   - Ownership percentages
   - Waiver wire adds
   - Best ball ADP
   - Free API, no auth required
   - Endpoint: https://docs.sleeper.com

### Medium Priority (Advanced Analytics)

4. **PFF (Pro Football Focus) Ingestion**
   - Coverage matchups (CB vs WR)
   - Man vs zone splits
   - Slot/shadow coverage
   - **Paid API** - Requires subscription
   - Alternative: Scrape free PFF articles (limited)

5. **ESPN Analytics Ingestion**
   - Pass block win rate
   - Run block win rate
   - Adjusted line yards
   - Free on ESPN.com
   - May require web scraping

6. **FantasyPros Advanced Stats**
   - Air yards (alternative to Next Gen)
   - Red zone usage
   - O-line rankings
   - Free tier available
   - Endpoint: https://www.fantasypros.com/api/

7. **KeepTradeCut API**
   - Dynasty trade values
   - Dynasty ADP
   - Market sentiment
   - Free public data
   - Endpoint: https://keeptradecut.com

### Low Priority (Nice to Have)

8. **FantasyPoints.com Coverage Data**
   - CB matchups
   - Coverage grades
   - **Paid API**

9. **Rotowire Injury Data**
   - Injury reports
   - Recovery timelines
   - Practice participation

10. **DraftKings/FanDuel Pricing**
    - DFS ownership projections
    - Salary/value metrics
    - Contest types

---

## Data Source Issues

### ⚠️ Currently Failing:

1. **Fantasy Football Data Pros**
   - Error: `ConnectionError: Name or service not known`
   - Issue: API endpoint may be down or incorrect
   - URL tested: `https://api.fantasyfootballdatapros.com/api/players/2024/week/18`
   - Action needed: Verify correct endpoint or find alternative

### ✅ Working Sources:
- nflverse (via nfl_data_py)
- ESPN (if public API exists)
- TheSportsDB
- WeatherAPI.com
- OpenWeatherMap
- WorldWeatherOnline

In [0]:
# Create recommendation table for source selection
print("Source Priority Recommendations")
print("="*70)

recommendations = [
    {
        'metric_category': 'Basic Stats',
        'primary': 'nflverse',
        'backup': 'espn',
        'confidence': 'High',
        'reason': 'Free, comprehensive, actively maintained'
    },
    {
        'metric_category': 'Air Yards',
        'primary': 'NextGenStats (NOT BUILT)',
        'backup': 'FantasyPros',
        'confidence': 'Medium',
        'reason': 'NextGen has official NFL tracking data'
    },
    {
        'metric_category': 'Red Zone Usage',
        'primary': 'NextGenStats (NOT BUILT)',
        'backup': 'nflverse',
        'confidence': 'Medium',
        'reason': 'Need detailed snap/touch data'
    },
    {
        'metric_category': 'Pace & Game Script',
        'primary': 'nflverse',
        'backup': 'None needed',
        'confidence': 'High',
        'reason': 'Play-by-play data already available'
    },
    {
        'metric_category': 'Coverage Matchups',
        'primary': 'PFF (NOT BUILT)',
        'backup': 'FantasyPoints',
        'confidence': 'Low',
        'reason': 'Requires paid PFF subscription'
    },
    {
        'metric_category': 'O-Line & Pressure',
        'primary': 'ESPN (NOT BUILT)',
        'backup': 'FantasyPros',
        'confidence': 'Medium',
        'reason': 'ESPN has free metrics published'
    },
    {
        'metric_category': 'Player Tracking',
        'primary': 'NextGenStats (NOT BUILT)',
        'backup': 'None',
        'confidence': 'High',
        'reason': 'NFL official tracking data'
    },
    {
        'metric_category': 'Betting Lines',
        'primary': 'The Odds API (NOT BUILT)',
        'backup': 'None',
        'confidence': 'High',
        'reason': 'Free tier 500 requests/month'
    },
    {
        'metric_category': 'Ownership & ADP',
        'primary': 'Sleeper API (NOT BUILT)',
        'backup': 'KeepTradeCut',
        'confidence': 'High',
        'reason': 'Free, real-time ownership data'
    },
    {
        'metric_category': 'Coaching Tendencies',
        'primary': 'nflverse',
        'backup': 'None needed',
        'confidence': 'High',
        'reason': 'Play-by-play includes all decisions'
    },
    {
        'metric_category': 'Weather',
        'primary': 'WeatherAPI.com',
        'backup': 'OpenWeatherMap',
        'confidence': 'High',
        'reason': '1M free calls/month vs 1K/day'
    }
]

rec_df = pd.DataFrame(recommendations)
print("\n")
display(rec_df)

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
not_built = rec_df[rec_df['primary'].str.contains('NOT BUILT')]
print(f"\n❗ Ingestion notebooks needed: {len(not_built)}")
print(f"\nPriority order:")
for idx, row in not_built.iterrows():
    print(f"  {idx+1}. {row['metric_category']}: {row['primary'].replace(' (NOT BUILT)', '')}")

In [0]:
%sql
-- Check for null/missing data in key fields
WITH source_quality AS (
  SELECT 
    source,
    COUNT(*) as total_records,
    COUNT(CASE WHEN fantasy_points IS NULL THEN 1 END) as null_fantasy_points,
    COUNT(CASE WHEN fantasy_points = 0 THEN 1 END) as zero_fantasy_points,
    COUNT(CASE WHEN player_name IS NULL OR player_name = '' THEN 1 END) as missing_names,
    COUNT(CASE WHEN position IS NULL OR position = '' THEN 1 END) as missing_positions,
    COUNT(CASE WHEN team IS NULL OR team = '' THEN 1 END) as missing_teams
  FROM main.fantasai.silver_weekly_stats
  WHERE season = 2024 AND week = 18
  GROUP BY source
)
SELECT 
  source,
  total_records,
  ROUND(100.0 * null_fantasy_points / total_records, 2) as pct_null_points,
  ROUND(100.0 * zero_fantasy_points / total_records, 2) as pct_zero_points,
  ROUND(100.0 * missing_names / total_records, 2) as pct_missing_names,
  ROUND(100.0 * missing_positions / total_records, 2) as pct_missing_pos,
  ROUND(100.0 * missing_teams / total_records, 2) as pct_missing_teams
FROM source_quality
ORDER BY total_records DESC

## Next Steps

### Immediate Actions

1. **✅ Test All Existing Ingestion Notebooks**
   - Run each notebook (10-20) individually
   - Check `silver_weekly_stats` table for data
   - Document which sources work vs fail

2. **❗ Fix Fantasy Football Data Pros**
   - Verify correct API endpoint
   - Check if service still exists
   - Find alternative if needed

3. **✅ Prioritize Missing Ingestion Notebooks**
   - **Must-have:** NextGenStats, The Odds API, Sleeper
   - **Nice-to-have:** PFF, ESPN Analytics, FantasyPros
   - **Optional:** KeepTradeCut, FantasyPoints

### Development Priorities

#### Phase 1: Core Data (Week 1-2)
- [ ] Create NextGenStats ingestion
- [ ] Create The Odds API ingestion
- [ ] Create Sleeper API ingestion
- [ ] Test all existing notebooks
- [ ] Document what each source provides

#### Phase 2: Advanced Metrics (Week 3-4)
- [ ] Create ESPN Analytics scraper
- [ ] Create FantasyPros advanced stats
- [ ] Create KeepTradeCut ingestion
- [ ] Build metric aggregation layer

#### Phase 3: Premium/Paid Sources (Week 5+)
- [ ] Evaluate PFF subscription
- [ ] Evaluate FantasyPoints API
- [ ] Build coverage matchup features

### Weekly Maintenance

**Every Tuesday (after games):**
1. Run all working ingestion notebooks
2. Check data quality (this notebook)
3. Validate source counts match expectations
4. Update CloudFlare R2 data for API

**Every Friday (before weekend):**
1. Run weather ingestion for upcoming games
2. Fetch betting lines (The Odds API)
3. Check injury updates
4. Generate weekly projections

### Automation Strategy

**Databricks Workflows:**
- Schedule ingestion notebooks as jobs
- Tuesday 8AM: Run all stat ingestions
- Saturday 6PM: Run weather + betting lines
- Daily: Update ownership data (Sleeper)

**CloudFlare Workers:**
- Fetch latest from R2 on API requests
- Cache for 30-60 minutes
- Refresh data weekly from Databricks